In [ ]:
import json
import numpy as np
import os
from scipy.stats import hypergeom
from sklearn.metrics import f1_score

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)


def petersen_random_baseline(A_est: np.ndarray) -> np.ndarray:
    A_est = A_est.astype(int)
    n = A_est.shape[0]

    m_est = int(np.sum(A_est))


    possible_edges = [(i, j) for i in range(n) for j in range(n) if i != j]
    m_max = len(possible_edges)

    if m_est > m_max:
        raise ValueError("Estimator predicted more edges than possible.")

    chosen_idx = np.random.choice(m_max, m_est, replace=False)

    A_rand = np.zeros((n, n), dtype=int)
    for idx in chosen_idx:
        i, j = possible_edges[idx]
        A_rand[i, j] = 1

    return A_rand

def compute_f1(A_true, A_est):
    mask = ~np.eye(A_true.shape[0], dtype=bool)
    y_true = A_true[mask].flatten()
    y_pred = A_est[mask].flatten()
    return f1_score(y_true, y_pred)


def compute_tnr(A_true, A_est):
    mask = ~np.eye(A_true.shape[0], dtype=bool)
    y_true = A_true[mask].flatten()
    y_pred = A_est[mask].flatten()

    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))

    if TN + FP == 0:
        return 1.0
    return TN / (TN + FP)


def petersen_p_value(A_true: np.ndarray, A_est: np.ndarray) -> float:
    n = A_true.shape[0]
    mask = ~np.eye(n, dtype=bool)

    true_flat = A_true[mask].astype(int)
    est_flat = A_est[mask].astype(int)

    m_max = len(true_flat)
    m_true = int(np.sum(true_flat))
    m_est = int(np.sum(est_flat))
    TP_obs = int(np.sum((true_flat == 1) & (est_flat == 1)))

    return hypergeom.sf(TP_obs - 1, m_max, m_true, m_est)


def run_petersen_folder(path):
    f1_scores = []
    tnr_scores = []
    f1_random_scores = []
    tnr_random_scores = []
    pvals = []

    for file in os.listdir(path):
        if not file.endswith(".json"):
            continue

        data = load_json(os.path.join(path, file))

        A_true = np.array(data["cm"], dtype=int)
        A_est = np.array(data["cm_est_freq"], dtype=int)

        # Generate Petersen random baseline
        A_rand = petersen_random_baseline(A_est)
        #A_rand = petersen_random_baseline(A_true).  #test with true graph
 
        # Estimator metrics
        f1_est = compute_f1(A_true, A_est)
        tnr_est = compute_tnr(A_true, A_est)

        # Random baseline metrics
        f1_rand = compute_f1(A_true, A_rand)
        tnr_rand = compute_tnr(A_true, A_rand)

        # p-value
        pv = petersen_p_value(A_true, A_est)

        f1_scores.append(f1_est)
        tnr_scores.append(tnr_est)
        f1_random_scores.append(f1_rand)
        tnr_random_scores.append(tnr_rand)
        pvals.append(pv)

        #print(f"\nFile: {file}")
        #print(f"  F1 (estimator):       {f1_est:.4f}")
        #print(f"  TNR (estimator):      {tnr_est:.4f}")
        #print(f"  F1 (random baseline): {f1_rand:.4f}")
        #print(f"  TNR (random baseline):{tnr_rand:.4f}")
        #print(f"  p-value:              {pv:.6f}")

    pvals = np.array(pvals)

    num_p = len(pvals)
    num_significant = np.sum(pvals < 0.05)
    frac_significant = num_significant / num_p if num_p > 0 else 0.0
    median_p = np.median(pvals)

    print("\n====================")
    print("      FINAL SUMMARY ")
    print("====================")
    print(path)
    print(f"Mean F1 (estimator):        {np.mean(f1_scores):.2f} \pm  {np.std(f1_scores):.2f}")
    print(f"Mean TNR (estimator):       {np.mean(tnr_scores):.2f} \pm {np.std(tnr_scores):.2f}")
    print(f"Mean F1 (random baseline):  {np.mean(f1_random_scores):.2f} \pm {np.std(f1_random_scores):.2f}")
    print(f"Mean TNR (random baseline): {np.mean(tnr_random_scores):.2f} \pm {np.std(tnr_random_scores):.2f}")
    print("")
    print(f"Total p-values:        {num_p}")
    print(f"Significant (p < 0.05): {num_significant}")
    print(f"Fraction significant:   {frac_significant:.4f}")
    print(f"Median p-value:         {median_p:.6f}")

    return {
        "mean_f1": np.mean(f1_scores),
        "mean_tnr": np.mean(tnr_scores),
        "mean_f1_random": np.mean(f1_random_scores),
        "mean_tnr_random": np.mean(tnr_random_scores),
        "num_pvals": num_p,
        "num_significant": num_significant,
        "frac_significant": frac_significant,
        "median_p": median_p
    }



In [ ]:
run_petersen_folder("results/")